In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
import rlcard
import numpy as np
from agents.models import TienLenNet, TienLenLSTMNet
from agents.agents import HeuristicSearchAgent, TienLenRandomAgent, TrainedDenseAgent, TrainedLSTMAgent, NaiveTienLenAgent
from rlcard.agents.dmc_agent import DMCTrainer
from rlcard.envs.registration import register
from rlcard.agents.dmc_agent.model import DMCModel
from rlcard.agents.dmc_agent.model import DMCAgent
from rlcard.utils import tournament
import random


In [2]:
register(env_id='tien-len', entry_point='envs.tienlen_env:TienLenEnv')
env = rlcard.make('tien-len')


In [3]:
orig_init = DMCAgent.__init__
def new_init(self, *args, **kwargs):
    orig_init(self, *args, **kwargs)
    self.use_raw = True
DMCAgent.__init__ = new_init


In [4]:
use_accel = torch.accelerator.is_available()
seed = 1
lr = 1e-4
model = 'dense'

In [5]:
class DenseReplayBuffer:
    def __init__(self, capacity=50000):
        # We use deque to automatically "forget" the oldest moves
        self.buffer = deque(maxlen=capacity)

    def push(self, obs, action, payoff):
        """
        obs: Snapshot feature vector (159,)
        action: Action feature vector (60,)
        payoff: Final game reward (scalar)
        """
        # Store as tensors immediately to save conversion time during sampling
        self.buffer.append((
            torch.as_tensor(obs, dtype=torch.float32),
            torch.as_tensor(action, dtype=torch.float32),
            torch.as_tensor([payoff], dtype=torch.float32)
        ))

    def sample(self, batch_size):
        # Pick random transitions from the buffer
        samples = random.sample(self.buffer, min(len(self.buffer), batch_size))

        # Zip them into batches
        obses, actions, payoffs = zip(*samples)

        # Return stacked tensors for the Neural Network
        return (torch.stack(obses),
                torch.stack(actions),
                torch.stack(payoffs))

    def __len__(self):
        return len(self.buffer)

class LSTMReplayBuffer:
    def __init__(self, capacity=100000):
        self.buffer = deque(maxlen=capacity)

    def push(self, history, obs, action, payoff):
        self.buffer.append((history, obs, action, payoff))

    def sample(self, batch_size):
        # Samples a random batch from the buffer
        samples = random.sample(self.buffer, min(len(self.buffer), batch_size))
        # Group them into tensors
        histories, obses, actions, payoffs = zip(*samples)
        return (torch.stack(histories),
                torch.stack(obses),
                torch.stack(actions),
                torch.stack(payoffs))

    def __len__(self):
        return len(self.buffer)

In [6]:
# 1. Setup the Model
# State is 159, Action is 52
torch.manual_seed(seed)

if use_accel:
    device = torch.accelerator.current_accelerator()
else:
    device = torch.device("cpu")



In [26]:
def train():
    env = rlcard.make('tien-len')
    trainer = DMCTrainer(
        env,
        cuda='',
        xpid='rl-tien-len',
        savedir='models',
        num_actors=5,
    )
    trainer.start()

def train_LSTM_iteration(model, optimizer, buffer, batch_size=512, episodes_per_step=10):
    model.train()

    # 1. Data Collection (Self-Play)
    for _ in range(episodes_per_step):
        state, player_id = env.reset()
        episode_data = []

        while not env.is_over():
            legal_actions = state['legal_actions']
            num_legal = len(legal_actions)

            # Prepare inputs for scoring
            obs_t = torch.FloatTensor(state['obs']).unsqueeze(0).repeat(num_legal, 1)
            hist_t = torch.FloatTensor(state['history']).unsqueeze(0).repeat(num_legal, 1, 1)
            act_list = list(legal_actions.keys())
            act_feats = torch.FloatTensor(np.array(list(legal_actions.values())))

            with torch.no_grad():
                # Score all moves
                q_values = model(hist_t, obs_t, act_feats).squeeze(-1)

                # Epsilon-Greedy exploration
                if random.random() < 0.1:
                    idx = random.randrange(num_legal)
                else:
                    idx = torch.argmax(q_values).item()

            chosen_action = act_list[idx]

            # Store transition details (converted to tensors)
            episode_data.append({
                'player_id': player_id,
                'history': torch.FloatTensor(state['history']),
                'obs': torch.FloatTensor(state['obs']),
                'act': torch.FloatTensor(legal_actions[chosen_action])
            })

            state, player_id = env.step(chosen_action)

        # Push episode to buffer with final payoffs
        final_payoffs = env.get_payoffs()
        for d in episode_data:
            buffer.push(d['history'], d['obs'], d['act'], torch.FloatTensor([final_payoffs[d['player_id']]]))

    # 2. Model Update (Batch from Buffer)
    if len(buffer) < batch_size:
        return 0 # Wait until buffer is full enough

    hist_b, obs_b, act_b, target_b = buffer.sample(batch_size)

    optimizer.zero_grad()
    predictions = model(hist_b, obs_b, act_b)
    loss = torch.nn.MSELoss()(predictions, target_b)
    loss.backward()
    optimizer.step()

    return loss.item()

def train_dense_iteration(model, optimizer, buffer, batch_size=512, episodes_per_step=10):
    model.train()

    # 1. Data Collection
    for _ in range(episodes_per_step):
        state, player_id = env.reset()
        episode_data = []

        while not env.is_over():
            legal_actions = state['legal_actions']
            num_legal = len(legal_actions)

            obs_t = torch.FloatTensor(state['obs']).unsqueeze(0).repeat(num_legal, 1)
            act_list = list(legal_actions.keys())
            act_feats = torch.FloatTensor(np.array(list(legal_actions.values())))

            with torch.no_grad():
                q_values = model(obs_t, act_feats).squeeze(-1)
                idx = torch.argmax(q_values).item() if random.random() > 0.1 else random.randrange(num_legal)

            chosen_action = act_list[idx]
            episode_data.append({
                'player_id': player_id,
                'obs': torch.FloatTensor(state['obs']),
                'act': torch.FloatTensor(legal_actions[chosen_action])
            })
            state, player_id = env.step(chosen_action)

        final_payoffs = env.get_payoffs()
        for d in episode_data:
            # Buffer only stores Obs + Act + Payoff
            buffer.push(d['obs'], d['act'], torch.FloatTensor([final_payoffs[d['player_id']]]))

    # 2. Update Model
    if len(buffer) < batch_size: return 0

    obs_b, act_b, target_b = buffer.sample(batch_size) # Ignore history

    optimizer.zero_grad()
    predictions = model(obs_b, act_b)
    loss = torch.nn.MSELoss()(predictions, target_b)
    loss.backward()
    optimizer.step()

    return loss.item()


def load_model():
    env = rlcard.make('tien-len')

    # 2. Re-create the model structure and load weights
    # Ensure state_shape and action_shape match your Env
    model = DMCModel(env.state_shape, [52])
    checkpoint = torch.load('models/rl-tien-len/model.tar', map_location='cpu')
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval() # Set to evaluation mode

    return TrainedAgent(model)


def evaluate(trained_agent, num=10):
    env = rlcard.make('tien-len')
    env.set_agents([trained_agent, NaiveTienLenAgent(), NaiveTienLenAgent(), NaiveTienLenAgent()])
    results = tournament(env, num)
    print(f"Tournament Results (Avg Payoffs): {results}")

def evaluate_with_winrates(trained_agent, num_games=1000):
    env = rlcard.make('tien-len')
    agents = [trained_agent, NaiveTienLenAgent(), NaiveTienLenAgent(), NaiveTienLenAgent()]

    wins = [0] * 4
    payoffs_sum = [0.0] * 4

    for _ in range(num_games):
        env.set_agents(agents)
        state, player_id = env.reset()

        while not env.is_over():
            agent = env.agents[player_id]
            action, _ = agent.eval_step(state)
            state, player_id = env.step(action)

        # Track who finished first (the one with empty hand)
        for idx, player in enumerate(env.game.players):
            payoffs_sum[idx] += env.get_payoffs()[idx]
            if not player.hand:  # First to empty = winner
                wins[idx] += 1
                break  # Only count first place

    for i in range(4):
        print(f"Player {i}: Win Rate = {wins[i]/num_games:.1%}, "
              f"Avg Payoff = {payoffs_sum[i]/num_games:.3f}")


def play_test_game(trained_agent):
    env = rlcard.make('tien-len')
    env.set_agents([trained_agent, NaiveTienLenAgent(), HeuristicSearchAgent(env.game.judger), TienLenRandomAgent()])

    state, player_id = env.reset()
    print("--- Game Start ---")

    while not env.is_over():
        # Get action from the current player's agent
        current_agent = env.agents[player_id]
        action, _ = current_agent.eval_step(state)

        # Display the play
        move_desc = action if action else "PASS"
        print(f"Player {player_id} plays: {move_desc}")

        # Advance environment
        state, player_id = env.step(action)

    print("--- Game Over ---")
    print(f"Final Payoffs: {env.get_payoffs()}")


In [8]:
if model == 'dense':
    model = TienLenNet(env.state_shape[0], [60])
    buffer = DenseReplayBuffer(capacity=10000)
    train_iteration = train_dense_iteration
    TrainedAgent = TrainedDenseAgent
else:
    model = TienLenLSTMNet(state_dim=159, action_dim=60)
    buffer = LSTMReplayBuffer(capacity=10000)
    train_iteration = train_LSTM_iteration
    TrainedAgent = TrainedLSTMAgent


optimizer = optim.AdamW(model.parameters(), lr=lr)
criterion = nn.MSELoss()


In [19]:
for i in range(1000):
    loss = train_iteration(model, optimizer, buffer)
    if i % 10 == 0:
        print(f"Iteration {i}, Loss: {loss:.4f}")
        agent = TrainedAgent(model)
        evaluate(agent)


Iteration 0, Loss: 0.7494
Tournament Results (Avg Payoffs): [np.float64(1.8), np.float64(1.7), np.float64(1.1), np.float64(1.4)]
Iteration 10, Loss: 0.7418
Tournament Results (Avg Payoffs): [np.float64(1.6), np.float64(1.6), np.float64(1.4), np.float64(1.4)]
Iteration 20, Loss: 0.6785
Tournament Results (Avg Payoffs): [np.float64(2.0), np.float64(1.0), np.float64(1.7), np.float64(1.3)]
Iteration 30, Loss: 0.7649
Tournament Results (Avg Payoffs): [np.float64(0.8), np.float64(1.7), np.float64(2.0), np.float64(1.5)]
Iteration 40, Loss: 1.1585
Tournament Results (Avg Payoffs): [np.float64(1.9), np.float64(1.3), np.float64(1.3), np.float64(1.5)]
Iteration 50, Loss: 0.7333
Tournament Results (Avg Payoffs): [np.float64(1.8), np.float64(1.2), np.float64(1.0), np.float64(2.0)]
Iteration 60, Loss: 0.8051
Tournament Results (Avg Payoffs): [np.float64(1.6), np.float64(2.0), np.float64(0.9), np.float64(1.5)]
Iteration 70, Loss: 1.3639
Tournament Results (Avg Payoffs): [np.float64(2.3), np.float64(1

In [10]:
# if not os.path.exists('models'): os.makedirs('models')
# train()
# agent = load_model()

In [27]:
agent = TrainedAgent(model)
evaluate_with_winrates(agent,1000)

Player 0: Win Rate = 86.6%, Avg Payoff = 1.873
Player 1: Win Rate = 13.4%, Avg Payoff = 0.333
Player 2: Win Rate = 0.0%, Avg Payoff = 0.000
Player 3: Win Rate = 0.0%, Avg Payoff = 0.000


In [ ]:
play_test_game(agent)